# SDK 대화와 스트리밍 — 수강생 실습본

원본: `src/raw/Chapter01/01-llm-api/02_openai.py`  
대응 슬라이드: `docs/raw/Chapter01/day01/1-2-development-environment-and-llm-api-mechanism.pdf`

**학습 목표:** 클라이언트 → 단일 질문 → 이전 응답을 잇는 대화 → 스트림의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** SDK가 LLM 서버에 단일 질문·연속 대화·스트리밍 요청을 보낸다. provider별 모델·인증과 여러 번의 호출을 확인한다.



## 흐름을 먼저 읽기

클라이언트 → 단일 질문 → 이전 응답을 잇는 대화 → 스트림

핵심은 요청과 응답의 필드 역할이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: 클라이언트 → 단일 질문 → 이전 응답을 잇는 대화 → 스트림. 두 번째 질문에 앞선 응답을 붙이지 않으면 어떻게 되나? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** SDK가 LLM 서버에 단일 질문·연속 대화·스트리밍 요청을 보낸다. provider별 모델·인증과 여러 번의 호출을 확인한다.



In [ ]:
import os
from collections import Counter
from pprint import pprint

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

provider = os.getenv('LLM_PROVIDER', 'samsung').strip().lower()
if provider == 'openai':
    model = os.getenv('OPENAI_MODEL')
    api_key = os.getenv('OPENAI_API_KEY')
    if not model or not api_key:
        raise RuntimeError('OPENAI_MODEL과 OPENAI_API_KEY가 필요합니다.')
    client = OpenAI(api_key=api_key)
elif provider == 'samsung':
    # 사내 게이트웨이 주소·모델·인증 헤더는 config/samsung.py에 모아 두었다.
    from config.samsung import llm_config
    samsung = llm_config()
    model = samsung['model']
    client = OpenAI(
        api_key='samsung-gateway',  # 실제 사내 인증은 x-dep-ticket 헤더에서 처리
        base_url=samsung['base_url'],
        default_headers=samsung['headers'],
    )
else:
    raise ValueError(f'지원하지 않는 LLM_PROVIDER: {provider}')


# OpenAI SDK 예제는 사외에서는 공식 API, 사내에서는 호환 게이트웨이를 사용한다.


# ── Single Turn ───────────────────────────────────────────────────────────────
# input에 문자열을 직접 전달하면 사용자 메시지로 처리
response = client.responses.create(
    model=model,
    input='LLM이 무엇인지 3문장으로 설명해줘.',
)
print(response.output_text)
print()


# ── Multi Turn ────────────────────────────────────────────────────────────────
# input에 역할(role)이 있는 메시지 리스트를 전달하면 대화 맥락 유지
# developer: 시스템 지시 / user: 사용자 입력 / assistant: AI 응답
context = [
    {'role': 'developer', 'content': '너는 단답형 답변 챗봇이야. 경어체를 유지하면서, 가능하면 단답형으로 답변해줘.'},
    {'role': 'user', 'content': '대한민국의 수도는 어디지?'},
]
response_1 = client.responses.create(model=model, input=context)
print(response_1.output_text)

# 이전 응답(response_1.output)을 context에 추가해 대화를 이어감
context += response_1.output
context += [{'role': 'user', 'content': '그곳의 인구는 어느정도야?'}]
response_2 = client.responses.create(model=model, input=context)
print(response_2.output_text)


# ── Streaming ─────────────────────────────────────────────────────────────────
# stream()을 사용하면 LLM이 생성하는 토큰을 실시간으로 받을 수 있음
# output_text.delta 이벤트에 실제 텍스트 조각이 담겨 있음
with client.responses.stream(
    model=model,
    input='LLM이 무엇인지 3문장으로 설명해줘.',
) as stream:
    event_types = []
    for event in stream:
        event_types.append(event.type)
        if 'output_text.delta' in event.type:
            print(event.delta, end='')
    print()
    pprint(Counter(event_types))


# ── Temperature & Top P (참고용) ──────────────────────────────────────────────
# temperature: 0에 가까울수록 일관된 답변, 높을수록 창의적(최대 2.0)
# top_p: 확률 상위 p% 토큰만 사용. temperature와 동시 조정은 권장하지 않음
# 주의: gpt-5 계열은 temperature 미지원
"""
response = client.responses.create(
    model=model,
    input='LLM이 무엇인지 3문장으로 설명해줘.',
    temperature=0.2,
    top_p=0.1,
)
print(response.output_text)
"""


## 결과를 검토하기

- 예상 동작: 단일 답변, 앞선 질문을 잇는 답변, 스트리밍 조각이 순서대로 출력된다.
- 확인 질문: 두 번째 질문에 앞선 응답을 붙이지 않으면 어떻게 되나?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
